In [ ]:
import json
import pandas as pd

from emu_renewal.constants import DATA_PATH
from emu_renewal.inputs import get_oxcgrt_data
from emu_renewal.outputs import add_bool_row_to_table
from emu_renewal.selection import gather_who_data, find_absent_inds, \
    find_neg_inds, find_outliers, find_nans_repeats, get_mob_avail_countries
from emu_renewal.utils import get_country_name

In [ ]:
analysis = "oxcgrt"

oxcgrt_countries = get_oxcgrt_data()["CountryCode"].unique()
mob_countries = get_mob_avail_countries()
countries = oxcgrt_countries if analysis == "oxcgrt" else mob_countries

In [ ]:
summary = pd.DataFrame(index=countries)
death_data, case_data = gather_who_data(countries)
no_deaths, no_cases = find_absent_inds(death_data, case_data, summary)
neg_deaths, neg_cases = find_neg_inds(death_data, case_data, summary)
death_outliers, case_outliers = find_outliers(death_data, case_data, summary)
death_nans, case_nans, death_reps, case_reps = find_nans_repeats(death_data, case_data, summary)

In [ ]:
excluded = set(no_deaths + no_cases + neg_deaths + neg_cases + death_nans + case_nans + death_reps + case_reps + death_outliers + case_outliers)
included = [c for c in countries if c not in excluded]
add_bool_row_to_table(summary, included, "Included")

In [ ]:
summary.index = summary.index.map(get_country_name)
summary

In [ ]:
json.dump(included, open(DATA_PATH / f"config/{analysis}_included.json", "w"))